In [21]:
import scanpy as sc
import torch
import numpy as np
from cancerfoundation.model.model import CancerFoundation
from cancerfoundation.data.data_collator import AnnDataCollator
from cancerfoundation.data.dataset import SingleCellDataset
from torch.utils.data import DataLoader
import json

model_name = "my_init_4gpu_lrx2"

# Load model
with open(f"./save/{model_name}/vocab.json", "r") as f:
    vocab = json.load(f)

model = CancerFoundation.load_from_checkpoint(
    f'./save/{model_name}/epoch_epoch=14.ckpt', 
    vocab=vocab, 
    strict=True
)
model.eval()
model.cuda()

CancerFoundation(
  (criterion): MSE()
  (model): OptimizedModule(
    (_orig_mod): TransformerModule(
      (value_encoder): TheirContinuousValueEncoder(
        (dropout): Dropout(p=0.2, inplace=False)
        (linear1): Linear(in_features=1, out_features=256, bias=True)
        (activation): ReLU()
        (linear2): Linear(in_features=256, out_features=256, bias=True)
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (criterion_conditions): CrossEntropyLoss()
      (criterion): MSE()
      (condition_encoders): ModuleDict(
        (technology): ConditionEncoder(
          (embedding): Embedding(15, 256)
          (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
      )
      (transformer_encoder): CFGenerator(
        (layers): ModuleList(
          (0-5): 6 x CFLayer(
            (self_attn): MHA(
              (self_attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=256, o

In [22]:
# Load data and filter to common genes
data = sc.read_h5ad("./premade_data/expr.h5ad")
common_genes = list(set(vocab.keys()).intersection(set(data.var.index)))
data = data[:, common_genes].copy()

sc.pp.highly_variable_genes(
    data, 
    n_top_genes=1200, 
    subset=False, # Do not filter yet, just compute and store in .var
    layer=None # Operates on data.X
)
data = data[:, data.var['highly_variable']]

# Prepare tensors manually
gene_ids = torch.LongTensor([vocab[g] for g in data.var.index])
count_matrix = data.X if isinstance(data.X, np.ndarray) else data.X.toarray()

# Embed in batches
batch_size = 64


In [23]:
embeddings = []
with torch.no_grad():
    
    for i in range(0, len(data), batch_size):
        batch_expr = torch.FloatTensor(count_matrix[i:i+batch_size]).cuda()
        batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()
        
        # Add CLS token
        cls_id = vocab["<cls>"]
        batch_genes = torch.cat([
            torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
            batch_genes
        ], dim=1)
        batch_expr = torch.cat([
            torch.full((batch_expr.shape[0], 1), -2, device='cuda'),  # pad_value for CLS
            batch_expr
        ], dim=1)
        
        # Create padding mask (False = not padded)
        padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')
        
        # Get embeddings from transformer
        emb = model.model.embed(
            src=batch_genes,
            values=batch_expr,
            src_key_padding_mask=padding_mask
        )[0]
        
        # Extract CLS token embedding (cell embedding)
        cell_emb = emb[:, 0, :]
        embeddings.append(cell_emb.cpu().numpy())

data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)
print(f"Embedded {data.shape[0]} cells with dimension {data.obsm['CancerFoundation'].shape[1]}")

Embedded 941 cells with dimension 256


/tmp/ipykernel_1009240/3515665783.py:33: ImplicitModificationWarning: Setting element `.obsm['CancerFoundation']` of view, initializing view as actual.
  data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)


In [24]:
import pandas as pd

In [25]:
embeddings = pd.DataFrame(data.obsm["CancerFoundation"], index=data.obs_names, columns=[f"dim_{i}" for i in range(data.obsm["CancerFoundation"].shape[1])])

In [26]:
data.write_h5ad(f"./embeddings/{model_name}_embeddings.h5ad")
embeddings.to_csv(f"./embeddings/{model_name}_embeddings.csv")